# Notebook 2: End-to-End QLoRA Fine-Tuning on LLaMA 2

## 1. Objective
This notebook implements a practical training pipeline:
- Load LLaMA 2 with 4-bit quantization
- Attach LoRA adapters
- Train with TRL SFTTrainer
- Evaluate and run inference
- Save adapter checkpoints

## 2. Training Workflow
1. Import libraries
2. Set model + tokenizer
3. Quantization config
4. LoRA config
5. Dataset prep
6. Training args
7. Trainer run
8. Save + infer

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 3. Model and Access
### 3.1 Hugging Face Access
You need gated access approval for official LLaMA 2 checkpoints.

### 3.2 Model Choice
Use a small variant first for debugging and pipeline checks.

In [ ]:
# Replace with your approved model id
model_name = "meta-llama/Llama-2-7b-hf"
# os.environ["HF_TOKEN"] = "your_token"

## 4. Quantization (QLoRA Core)
4-bit NF4 quantization reduces memory while keeping trainability through LoRA layers.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
print(bnb_config)

## 5. Load Tokenizer and Model
If you are CPU-only, keep this section as reference and run on a GPU runtime.

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
# model = prepare_model_for_kbit_training(model)
print("Model loading cell prepared. Uncomment for real training.")

## 6. LoRA Adapter Configuration
### Important Hyperparameters
- r: rank
- lora_alpha: scaling
- lora_dropout: regularization
- target_modules: attention projections to adapt

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
print(lora_config)

## 7. Build Training Dataset
Use your own curated data in production. This is a minimal instructional sample.

In [ ]:
samples = [
    {"text": "<s>[INST] Define overfitting. [/INST] Overfitting is when a model learns noise from training data and performs poorly on new data. </s>"},
    {"text": "<s>[INST] What is LoRA? [/INST] LoRA is a parameter-efficient tuning method that learns low-rank updates while freezing base weights. </s>"},
    {"text": "<s>[INST] Why use QLoRA? [/INST] QLoRA reduces memory usage by quantizing base model weights while training lightweight adapters. </s>"}
]
dataset = Dataset.from_list(samples)
print(dataset)
print(dataset[0]["text"][:120] + "...")

## 8. Training Arguments
### Recommended Starter Values
- LR: 2e-4 for LoRA
- Epochs: 1-3 for pilot
- Batch size: small + gradient accumulation

In [ ]:
training_args = TrainingArguments(
    output_dir="./llama2_qlora_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=50,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)
print(training_args)

## 9. Trainer Setup
When ready on GPU, uncomment and run.

In [ ]:
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=dataset,
#     peft_config=lora_config,
#     dataset_text_field="text",
#     max_seq_length=512,
#     tokenizer=tokenizer,
#     args=training_args
# )
# trainer.train()
print("Trainer cell prepared.")

## 10. Save Adapters and Tokenizer
Saving adapters keeps storage small and reusable across environments.

In [ ]:
# adapter_dir = "./llama2_qlora_adapter"
# trainer.model.save_pretrained(adapter_dir)
# tokenizer.save_pretrained(adapter_dir)
print("Save cell prepared.")

## 11. Inference Test Template

In [ ]:
# prompt = "<s>[INST] Explain bias-variance tradeoff in simple words. [/INST]"
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# with torch.no_grad():
#     out = model.generate(**inputs, max_new_tokens=120, temperature=0.7, top_p=0.9)
# print(tokenizer.decode(out[0], skip_special_tokens=True))
print("Inference template ready.")

## 12. Notes and Common Issues
- OOM: reduce sequence length / batch size / use gradient checkpointing
- Loss not improving: lower LR, improve data quality, check prompt format
- Bad generation style: align inference template with training format

## 13. Next Step
Proceed to Notebook 3 for advanced optimization, evaluation, and deployment.